# Brecha salarial por género: vendedores y auxiliares de venta en tiendas,   almacenes y afines. (CIUO-08: 5223)

Análisis de diferencias significativas entre los promedios salariales de hombres y mujeres para el oficio 5223 (vendedores y auxiliares de venta en
tiendas, almacenes y afines) en los departamentos de Antioquia, Caldas, Quindío y Valle del Cauca, usando datos de la GEIH.

In [0]:
%pip install statsmodels -q

from pyspark.sql.functions import col, when, count, avg

import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
import matplotlib.pyplot as plt

# Departamentos objetivo: Antioquia=5, Caldas=17, Quindío=63, Valle del Cauca=76
DEPTOS = [5, 17, 63, 76]
DEPTO_NAMES = {5: "Antioquia", 17: "Caldas", 63: "Quindío", 76: "Valle del Cauca"}
OFICIO = 5223  # vendedores y auxiliares de venta en tiendas y almacenes

# Cargar tablas GEIH
cgse = spark.read.table("workspace.electiva.cgse").select(
    col("DIRECTORIO"), col("SECUENCIA_P"), col("ORDEN"),
    col("P3271").alias("sexo")
)

ocupados = spark.read.table("workspace.electiva.ocupados").select(
    col("DIRECTORIO"), col("SECUENCIA_P"), col("ORDEN"),
    col("OFICIO_C8"), col("P6500").alias("salario"),
    col("FEX_C18"), col("DPTO"), col("OCI")
)

# Join cgse + ocupados para obtener sexo, oficio, salario y departamento
df = ocupados.join(cgse, on=["DIRECTORIO", "SECUENCIA_P", "ORDEN"], how="inner")

# Filtrar: oficio 5223, 4 departamentos, ocupados, con salario y sexo válidos
df_5223 = df.filter(
    (col("DPTO").isin(DEPTOS)) &
    (col("OCI") == 1) &
    (col("OFICIO_C8") == OFICIO) &
    (col("salario").isNotNull()) &
    (col("sexo").isin([1, 2]))
).withColumn(
    "sexo_desc", when(col("sexo") == 1, "Hombre").otherwise("Mujer")
).withColumn(
    "depto_desc",
    when(col("DPTO") == 5, "Antioquia")
    .when(col("DPTO") == 17, "Caldas")
    .when(col("DPTO") == 63, "Quindío")
    .when(col("DPTO") == 76, "Valle del Cauca")
)

# Convertir a pandas para análisis estadístico
pdf = df_5223.select("salario", "sexo_desc", "depto_desc", "FEX_C18").toPandas()
print(f"Total de registros: {len(pdf)}")
print(f"Hombres: {len(pdf[pdf['sexo_desc']=='Hombre'])}, Mujeres: {len(pdf[pdf['sexo_desc']=='Mujer'])}")
print(f"Departamentos: {pdf['depto_desc'].unique().tolist()}")
display(pdf.head(10))

In [0]:
# Tabla de promedios salariales por departamento y género
resumen = pdf.groupby(["depto_desc", "sexo_desc"]).agg(
    n=("salario", "count"),
    promedio=("salario", "mean"),
    mediana=("salario", "median"),
    desv_est=("salario", "std"),
    minimo=("salario", "min"),
    maximo=("salario", "max")
).round(0).reset_index()

print("=== PROMEDIO SALARIAL POR DEPARTAMENTO Y GÉNERO ===")
print("(Oficio 5223: Dependientes y vendedores de tiendas y almacenes)\n")
display(resumen)

# Diferencia porcentual por departamento
print("\n=== DIFERENCIA PROMEDIO (Hombre - Mujer) POR DEPARTAMENTO ===")
for depto in ["Antioquia", "Caldas", "Quindío", "Valle del Cauca"]:
    h = pdf[(pdf['depto_desc']==depto) & (pdf['sexo_desc']=='Hombre')]['salario'].mean()
    m = pdf[(pdf['depto_desc']==depto) & (pdf['sexo_desc']=='Mujer')]['salario'].mean()
    diff = h - m
    pct = (diff / m) * 100
    print(f"{depto:20s}: Hombre=${h:>12,.0f}  Mujer=${m:>12,.0f}  Diferencia=${diff:>12,.0f} ({pct:+.1f}%)")

# Promedios generales
h_all = pdf[pdf['sexo_desc']=='Hombre']['salario'].mean()
m_all = pdf[pdf['sexo_desc']=='Mujer']['salario'].mean()
print(f"\n{'TOTAL':20s}: Hombre=${h_all:>12,.0f}  Mujer=${m_all:>12,.0f}  Diferencia=${h_all-m_all:>12,.0f} ({(h_all-m_all)/m_all*100:+.1f}%)")

In [0]:
# --- ANOVA DE DOS VÍAS: salario ~ sexo + departamento + sexo:departamento ---
# Esta prueba evalúa simultáneamente el efecto del género, del departamento y su interacción

modelo = ols("salario ~ C(sexo_desc) * C(depto_desc)", data=pdf).fit()
anova_table = sm.stats.anova_lm(modelo, typ=2)

print("=== ANOVA DE DOS VÍAS (salario ~ sexo * departamento) ===\n")
print(anova_table.round(4))
print()
alpha = 0.05
for factor in anova_table.index:
    p_val = anova_table.loc[factor, 'PR(>F)']
    sig = "SIGNIFICATIVO" if p_val < alpha else "NO significativo"
    print(f"  {factor:45s}: p-value = {p_val:.4f}  →  {sig}")

In [0]:
# --- PRUEBAS t DE STUDENT POR DEPARTAMENTO ---
# Compara si la diferencia de medias salariales entre hombres y mujeres es significativa
# en cada departamento individualmente

print("=== PRUEBA t DE STUDENT POR DEPARTAMENTO (Hombres vs Mujeres) ===\n")
print(f"{'Departamento':<20s} {'t-stat':>8s} {'p-value':>10s} {'Significativo?':>15s}  Hombres (n)       Mujeres (n)")
print("-" * 95)

resultados_t = []
for depto in ["Antioquia", "Caldas", "Quindío", "Valle del Cauca"]:
    h_data = pdf[(pdf['depto_desc']==depto) & (pdf['sexo_desc']=='Hombre')]['salario']
    m_data = pdf[(pdf['depto_desc']==depto) & (pdf['sexo_desc']=='Mujer')]['salario']
    
    # Prueba t de dos muestras independientes (Welch's t-test, no asume varianzas iguales)
    t_stat, p_val = stats.ttest_ind(h_data, m_data, equal_var=False)
    
    # Prueba de Levene para igualdad de varianzas
    lev_stat, lev_p = stats.levene(h_data, m_data)
    
    sig = "SÍ" if p_val < 0.05 else "NO"
    print(f"{depto:<20s} {t_stat:>8.3f} {p_val:>10.4f} {sig:>15s}  ${h_data.mean():>10,.0f} ({len(h_data):>3d})  ${m_data.mean():>10,.0f} ({len(m_data):>3d})")
    resultados_t.append({"depto": depto, "t_stat": t_stat, "p_val": p_val, "sig": sig})

# Prueba t global (todos los departamentos juntos)
h_all = pdf[pdf['sexo_desc']=='Hombre']['salario']
m_all = pdf[pdf['sexo_desc']=='Mujer']['salario']
t_global, p_global = stats.ttest_ind(h_all, m_all, equal_var=False)
sig_global = "SÍ" if p_global < 0.05 else "NO"
print("-" * 95)
print(f"{'GLOBAL':<20s} {t_global:>8.3f} {p_global:>10.4f} {sig_global:>15s}  ${h_all.mean():>10,.0f} ({len(h_all):>3d})  ${m_all.mean():>10,.0f} ({len(m_all):>3d})")

In [0]:
# --- VISUALIZACIÓN: Boxplot de salarios por departamento y género ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Boxplot por departamento y género
deptos_order = ["Antioquia", "Caldas", "Quindío", "Valle del Cauca"]
data_box = []
labels = []
for d in deptos_order:
    for s in ["Hombre", "Mujer"]:
        data_box.append(pdf[(pdf['depto_desc']==d) & (pdf['sexo_desc']==s)]['salario'].values)
        labels.append(f"{d[:4]}\n{s[:1]}")

bp = axes[0].boxplot(data_box, labels=labels, patch_artist=True)
colors = ['#4C72B0', '#DD8452'] * 4
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
axes[0].set_title("Distribución salarial por departamento y género\n(Oficio 5223)", fontsize=12)
axes[0].set_ylabel("Salario mensual (COP)", fontsize=11)
axes[0].tick_params(axis='x', labelsize=9)

# Gráfico de barras de promedios
x = np.arange(len(deptos_order))
width = 0.35
h_means = [pdf[(pdf['depto_desc']==d) & (pdf['sexo_desc']=='Hombre')]['salario'].mean() for d in deptos_order]
m_means = [pdf[(pdf['depto_desc']==d) & (pdf['sexo_desc']=='Mujer')]['salario'].mean() for d in deptos_order]

axes[1].bar(x - width/2, h_means, width, label='Hombres', color='#4C72B0')
axes[1].bar(x + width/2, m_means, width, label='Mujeres', color='#DD8452')
axes[1].set_title("Promedio salarial por departamento y género\n(Oficio 5223)", fontsize=12)
axes[1].set_ylabel("Salario promedio (COP)", fontsize=11)
axes[1].set_xticks(x)
axes[1].set_xticklabels(deptos_order, fontsize=10)
axes[1].legend()
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v/1e6:.1f}M'))

plt.tight_layout()
display(fig)

## Conclusiones

**Oficio analizado:** 5223 — vendedores y auxiliares de venta en
tiendas, almacenes y afines
**Datos:** 271 registros de la GEIH (106 hombres, 165 mujeres) en Antioquia, Caldas, Quindío y Valle del Cauca.

### Resultados de las pruebas estadísticas

1. **ANOVA de dos vías** (salario ~ sexo × departamento): Evalúa si el género, el departamento o su interacción tienen un efecto significativo sobre el salario. Revise los p-values de la tabla ANOVA.

2. **Pruebas t por departamento**: Compara hombres vs mujeres dentro de cada departamento. Un p-value < 0.05 indica diferencia significativa.

### Interpretación

- Si el p-value del factor **sexo** en el ANOVA es ≥ 0.05, **no existe evidencia estadísticamente significativa** de una diferencia salarial general entre hombres y mujeres para este oficio.
- Si el p-value del factor **departamento** es < 0.05, los salarios difieren significativamente entre departamentos.
- Si la **interacción sexo×departamento** es significativa, la brecha salarial depende del departamento.
- Las pruebas t individuales muestran en qué departamentos específicos la diferencia es significativa.

## (pruebas)

- Conclusión: ni el sexo, ni el departamento, ni su interacción tienen un efecto significativo sobre el salario, así que no hay evidencia de diferencia entre hombres y mujeres.
- ANOVA: sexo p = 0.6841; departamento p = 0.2700; interacción p = 0.0786.(Ninguno significativo)
- Pruebas t por departamento: ninguna es significativa (p entre 0.0889 y 0.4666).
- Global: hombres $1,728,464 y mujeres $1,690,318 (diferencia de $38,146, ≈ 2.2%), con p = 0.6638.

In [0]:
# --- VERIFICACIÓN DE SUPUESTOS ---

print("=== VERIFICACIÓN DE SUPUESTOS PARA ANOVA ===\n")

# 1. Normalidad (Shapiro-Wilk por grupo)
print("1. Prueba de normalidad (Shapiro-Wilk):")
for depto in ["Antioquia", "Caldas", "Quindío", "Valle del Cauca"]:
    for sexo in ["Hombre", "Mujer"]:
        data = pdf[(pdf['depto_desc']==depto) & (pdf['sexo_desc']==sexo)]['salario']
        if len(data) >= 3:
            stat, p = stats.shapiro(data)
            norm = "normal" if p > 0.05 else "no normal"
            print(f"  {depto:20s} {sexo:8s}: W={stat:.4f}, p={p:.4f} → {norm}")

# 2. Homogeneidad de varianzas (Levene)
print("\n2. Prueba de homogeneidad de varianzas (Levene entre géneros):")
h_all = pdf[pdf['sexo_desc']=='Hombre']['salario']
m_all = pdf[pdf['sexo_desc']=='Mujer']['salario']
lev_stat, lev_p = stats.levene(h_all, m_all)
homov = "homogéneas" if lev_p > 0.05 else "no homogéneas"
print(f"  Levene: W={lev_stat:.4f}, p={lev_p:.4f} → varianzas {homov}")

# 3. Alternativa no paramétrica: Mann-Whitney U global
print("\n3. Prueba no paramétrica (Mann-Whitney U) - global:")
u_stat, u_p = stats.mannwhitneyu(h_all, m_all, alternative='two-sided')
sig_u = "SIGNIFICATIVO" if u_p < 0.05 else "NO significativo"
print(f"  U={u_stat:.1f}, p={u_p:.4f} → {sig_u}")

print("\n4. Prueba no paramétrica (Mann-Whitney U) - por departamento:")
for depto in ["Antioquia", "Caldas", "Quindío", "Valle del Cauca"]:
    h_data = pdf[(pdf['depto_desc']==depto) & (pdf['sexo_desc']=='Hombre')]['salario']
    m_data = pdf[(pdf['depto_desc']==depto) & (pdf['sexo_desc']=='Mujer')]['salario']
    u_stat, u_p = stats.mannwhitneyu(h_data, m_data, alternative='two-sided')
    sig_u = "SÍ" if u_p < 0.05 else "NO"
    print(f"  {depto:20s}: U={u_stat:.1f}, p={u_p:.4f} → {sig_u} significativo")

## Conclusión final

**¿Existen diferencias significativas entre los promedios salariales de hombres y mujeres para el oficio 5223 en los departamentos de Caldas, Quindío, Antioquia y Valle del Cauca?**

Revise los resultados de las celdas anteriores para responder. En general:

- El ANOVA de dos vías indica si el efecto principal del **género** es significativo.
- Las pruebas t (y Mann-Whitney como respaldo no paramétrico) por departamento indican si la diferencia es significativa en cada departamento individual.
- Si ningún p-value es menor a 0.05, **no existe evidencia estadísticamente significativa** de diferencia salarial entre hombres y mujeres para este oficio en estos departamentos.
- Si alguno es menor a 0.05, **sí existe diferencia significativa** en ese departamento o globalmente.

## (conclusión final)

- Respuesta: No existe diferencia salarial estadísticamente significativa entre hombres y mujeres en el oficio 5223 en los cuatro departamentos.
- ANOVA: el sexo tiene p = 0.6841.
- Global: la diferencia de $38,146 no es significativa (t p = 0.6638; Mann-Whitney p = 0.8603).
- Por departamento: ninguna prueba t es significativa. Solo Quindío lo es en Mann-Whitney (p = 0.0270), a favor de las mujeres.